# 02 — Data Exploration

Exploración visual de las curvas de luz descargadas en `01_data_download.ipynb`.

Cargamos los datos desde `data/raw/`, aplicamos detección de outliers y visualizamos
los resultados para entender cómo luce un tránsito planetario real.

> Asegúrate de haber ejecutado `01_data_download.ipynb` antes de correr este notebook.

## 0. Imports y configuración

In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt

from exoplanets.data.loader import LightCurveLoader
from exoplanets.detection.statistical import OutlierDetector
from exoplanets.visualization.plots import plot_lightcurve, plot_score_distribution

loader = LightCurveLoader()

# Verificar que existen los datos
archivos = sorted(os.listdir('../data/raw'))
print(f' Archivos disponibles en data/raw/: {len(archivos)}')
for f in archivos:
    print(f' doc  {f}')

## 1. Cargar curva de luz — Kepler-11

In [ ]:
lc = loader.fetch_from_csv('../data/raw/kepler_11_q1_raw.csv')

print(f'Puntos de datos : {len(lc)}')
print(f'Rango de tiempo : {lc["time"].min():.2f} — {lc["time"].max():.2f} días')
print(f'Flujo promedio  : {lc["flux"].mean():.6f}')
print(f'Desv. estándar  : {lc["flux"].std():.6f}')
lc.head()

## 2. Visualización de la curva de luz original

In [ ]:
plot_lightcurve(lc, title='Kepler-11 — Trimestre 1 (sin detección)')

## 3. Detección de outliers con MAD

In [ ]:
detector = OutlierDetector(method='mad', threshold=4.0, direction='down')
mask = detector.fit_predict(lc)

print(f'Outliers detectados : {mask.sum()} de {len(lc)} puntos ({100*mask.mean():.2f}%)')

In [ ]:
plot_lightcurve(lc, outlier_mask=mask, title='Kepler-11 — Tránsitos Detectados (MAD)')

## 4. Distribución de scores

In [ ]:
plot_score_distribution(detector.scores, threshold=4.0)

## 5. Tabla de outliers detectados

In [ ]:
summary = detector.summary(lc, mask)
print(f'Top 10 dips más profundos:')
summary.head(10)

## 6. Comparación — estrella CON vs SIN planetas

In [ ]:
# Cargar estrella de control (sin planetas)
lc_control = loader.fetch_from_csv('../data/raw/kic_3114789_q1_raw.csv')

detector_ctrl = OutlierDetector(method='mad', threshold=4.0, direction='down')
mask_ctrl = detector_ctrl.fit_predict(lc_control)

# Graficar lado a lado
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.patch.set_facecolor('#0d1117')

for ax, lc_data, mask_data, title in [
    (axes[0], lc, mask, 'Kepler-11 (CON planetas) — Outliers detectados'),
    (axes[1], lc_control, mask_ctrl, 'KIC 3114789 (SIN planetas) — Outliers detectados'),
]:
    ax.set_facecolor('#0d1117')
    ax.plot(lc_data['time'], lc_data['flux'], color='#4a9eff', linewidth=0.6, alpha=0.8)
    if mask_data.any():
        ax.scatter(lc_data['time'][mask_data], lc_data['flux'][mask_data],
                   color='#ff4757', s=25, zorder=5, label=f'Dips ({mask_data.sum()})')
    ax.set_title(title, color='#e6edf3', fontsize=12)
    ax.set_ylabel('Flujo', color='#8b949e')
    ax.tick_params(colors='#8b949e')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')
    ax.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#e6edf3')

axes[1].set_xlabel('Tiempo (BKJD)', color='#8b949e', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Kepler-11     — outliers: {mask.sum()}')
print(f'KIC 3114789   — outliers: {mask_ctrl.sum()}')

## 7. Guardar resultados en data/processed/

In [ ]:
# Guardar curva con columna de outlier agregada
lc_resultado = lc.copy()
lc_resultado['is_outlier'] = mask
lc_resultado['score'] = detector.scores

lc_resultado.to_csv('../data/processed/kepler_11_q1_processed.csv', index=False)
print(' Guardado en data/processed/kepler_11_q1_processed.csv')
print(f'   Columnas: {list(lc_resultado.columns)}')
print(f'   Filas   : {len(lc_resultado)}')
print('\n  Continúa en 03_outlier_methods.ipynb')